# Decision Trees

**Companion lesson:** https://ml-viz.vercel.app/courses/knn-decision-trees/02-decision-trees

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Gini Impurity

$$\text{Gini}(p) = 1 - \sum_k p_k^2$$

In [ ]:
p = np.linspace(0, 1, 200)
gini = 1 - p**2 - (1 - p)**2
entropy = -(p * np.log2(p + 1e-10) + (1 - p) * np.log2(1 - p + 1e-10))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, gini, color='#818cf8', linewidth=2, label='Gini')
ax.plot(p, entropy / entropy.max() * 0.5, color='#14b8a6', linewidth=2, label='Entropy (scaled)')
ax.set_title('Impurity Measures', color='white')
ax.set_xlabel('p(class=1)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Gini at 60/40 split: {1 - 0.6**2 - 0.4**2:.3f}')

## Decision Tree from Scratch (simplified)

In [ ]:
class SimpleTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
    def _gini(self, y):
        p = np.mean(y)
        return 1 - p**2 - (1 - p)**2
    def _best_split(self, X, y):
        best_gain, best_feat, best_thresh = -1, 0, 0
        parent_gini = self._gini(y)
        for feat in range(X.shape[1]):
            for thresh in np.unique(X[:, feat]):
                left = y[X[:, feat] <= thresh]
                right = y[X[:, feat] > thresh]
                if len(left) == 0 or len(right) == 0: continue
                gini = (len(left) * self._gini(left) + len(right) * self._gini(right)) / len(y)
                gain = parent_gini - gini
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh
    def fit(self, X, y, depth=0):
        self.is_leaf = depth >= self.max_depth or len(np.unique(y)) == 1
        if self.is_leaf:
            self.pred = np.bincount(y).argmax()
            return
        f, t = self._best_split(X, y)
        self.feat, self.thresh = f, t
        left_mask = X[:, f] <= t
        self.left = SimpleTree(self.max_depth)
        self.right = SimpleTree(self.max_depth)
        self.left.fit(X[left_mask], y[left_mask], depth + 1)
        self.right.fit(X[~left_mask], y[~left_mask], depth + 1)
    def predict_one(self, x):
        if self.is_leaf: return self.pred
        return self.left.predict_one(x) if x[self.feat] <= self.thresh else self.right.predict_one(x)
    def predict(self, X): return np.array([self.predict_one(x) for x in X])

tree = SimpleTree(max_depth=4)
tree.fit(X, y)

Z = tree.predict(grid).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=20, alpha=0.7)
ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=20, alpha=0.7)
ax.set_title('Decision Tree (depth=4)', color='white')
plt.tight_layout()
plt.show()

## Information gain from a split

A split is scored by how much it reduces impurity: parent impurity minus the weighted impurity of the children.

In [ ]:
def gini(p):
    return 1 - sum(pi**2 for pi in p)

parent = gini([0.5, 0.5])                       # 50/50 -> 0.5
left   = gini([6/7, 1/7]); right = gini([1/3, 2/3])
weighted = (7/10) * left + (3/10) * right
print(f'parent={parent:.3f}, children weighted={weighted:.3f}, gain={parent-weighted:.3f}')

## Tree depth controls overfitting

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import cross_val_score

Xm, ym = make_moons(n_samples=300, noise=0.3, random_state=0)
for depth in [1, 3, 5, None]:
    s = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=0), Xm, ym, cv=5)
    print(f'max_depth={str(depth):>4}: CV accuracy = {s.mean():.3f}')

## Key takeaways

- Trees split greedily on the feature/threshold that most reduces **impurity** (Gini or entropy).
- **Information gain** = parent impurity − weighted child impurity.
- Unconstrained trees overfit; limit `max_depth` / `min_samples_leaf` or prune.
- Trees are interpretable and need no feature scaling, but are high-variance alone — hence ensembles.